# Αξιολόγηση & Παρουσίαση
Covers steps **5.1** and **5.3**.

| Step | Task |
|------|------|
| 5.1  | Performance metrics (Accuracy, FP, FN, Precision, Recall, F1) |
| 5.3  | Comparative charts (Entropy vs Time, k vs Performance) |

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

for candidate in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    if (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None and Path("/content/drive/MyDrive").exists():
    # Fast paths to check instead of slow rglob
    candidate_paths = [
        Path("/content/drive/MyDrive/Entropy-Based Network Intrusion Detection using Markov Chains"),
        Path("/content/drive/MyDrive/IT_PROJECT"),
        Path("/content/drive/MyDrive/IT_Project"),
    ]
    for p in candidate_paths:
        if (p / "config.yaml").exists():
            PROJECT_ROOT = p
            break

if PROJECT_ROOT is None:
    PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name in ["2_Data_Pipeline", "3_Training", "4_Detection", "5_Evaluation"] else NOTEBOOK_DIR

os.chdir(PROJECT_ROOT)
print(f"PROJECT_ROOT verified: {PROJECT_ROOT}")
DATA_PATH = PROJECT_ROOT / "data"


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## 5.1 – Μετρικές Απόδοσης

In [ ]:
import pandas as pd
import yaml
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt

RESULTS_DIR = PROJECT_ROOT / "5_Evaluation" / "results"
results_file = RESULTS_DIR / "detection_results.csv"

with open(PROJECT_ROOT / "config.yaml", "r") as f:
    config = yaml.safe_load(f)
K_ORDER = config["model"]["recommended_k"]

if results_file.exists():
    df = pd.read_csv(results_file)
    y_true = df["y_true"]
    y_pred = df["y_pred"]
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    try:
        roc_auc = roc_auc_score(y_true, df["Deviation"])
    except ValueError:
        roc_auc = 0.0
        
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    
    metrics = ['Precision', 'Recall', 'F1-Score']
    scores = [precision, recall, f1]
    
    plt.figure(figsize=(6, 4))
    plt.bar(metrics, scores, color=['#2b5c8f', '#4682b4', '#d9534f'])
    plt.ylim(0, 1.1)
    plt.ylabel('Score')
    plt.title(f'Detection Performance (k={K_ORDER})')
    for i, v in enumerate(scores):
        plt.text(i, v + 0.02, str(round(v, 3)), ha='center')
    plt.tight_layout()
    plt.show()
else:
    print("No detection results found! Please run detection.ipynb first.")


## 5.3 – Συγκριτικά Γραφήματα

In [ ]:
summary_path = PROJECT_ROOT / "3_Training" / "models" / "training_summary.csv"
df_summary = pd.read_csv(summary_path)

plt.figure(figsize=(8, 5))
features = ['TCP_Flags_Int_sym', 'Tot_Fwd_Pkts_sym', 'Flow_Duration_sym', 'TotLen_Fwd_Pkts_sym']

for feat in features:
    sub = df_summary[df_summary['feature'] == feat]
    if not sub.empty:
        plt.plot(sub['k'], sub['baseline_entropy'], marker='o', label=feat.replace('_sym', ''))

plt.xlabel('Markov Order (k)', fontsize=12)
plt.ylabel('Baseline Conditional Entropy (bits)', fontsize=12)
plt.title('Baseline Conditional Entropy Decay vs Markov Order (k)', fontsize=14)
plt.xticks([1, 2, 3])
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()